# Reproducing arXiv:2605.17307
**Deep Reinforcement Learning Framework for Diversified Portfolio Management Across Global Equity Markets**  
Kashif & Ślepaczuk (2026) — [arXiv:2605.17307](https://arxiv.org/abs/2605.17307)

Auto-generated by **ArXivist Stage 5**. This notebook is a *guided walkthrough* — it does **not** reproduce the full 16-fold × 5-config × 3-market study (that requires Bloomberg data and ~3 400 GPU-h). Instead it:

1. Shows the per-asset and global state-feature engineering (paper §5.1).
2. Walks through the reward decomposition (Eq. 16/17).
3. Builds the `PortfolioEnv` on **synthetic** price data.
4. Runs a 200-step SAC smoke test with the same hyperparameters as the paper.
5. Computes ARC / ASD / MD / IR2 (paper §5.8) on the synthetic out-of-sample episode.

---

## 0. Setup

In [ ]:
import sys, pathlib
REPO = pathlib.Path('..').resolve()
sys.path.insert(0, str(REPO / 'src'))

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from portfolio_rl.utils import load_config, seed_everything, resolve_device
seed_everything(42)
print('torch:', torch.__version__, '| cuda available:', torch.cuda.is_available())

## 1. State features (paper §5.1)

Per-asset features (15 total):
- log returns over 1, 5, 20, 60 days
- rolling volatility over 5, 20 days
- RSI(14), MACD-histogram(12,26,9), Bollinger %B(20,2σ)
- distance from 20-day high, mean-reversion vs 20-day MA
- 60-day rolling β vs market proxy, 20-day log return

Global features (7 total): VIX level, VIX 5-day change, cross-sectional mean return, cross-sectional 5d vol, market breadth, market-proxy 5d/20d returns.

In [ ]:
from portfolio_rl.data import FeatureBuilder

# Synthetic prices: 20 assets, 500 days
rng = np.random.default_rng(42)
dates = pd.date_range('2020-01-01', periods=500, freq='B')
tickers = [f'A{i:02d}' for i in range(20)]
rets = rng.normal(0.0005, 0.012, size=(500, 20))
prices = pd.DataFrame(100 * np.exp(np.cumsum(rets, axis=0)), index=dates, columns=tickers)
market = prices.mean(axis=1)
vix = pd.Series(15 + 5 * np.sin(np.linspace(0, 8, 500)) + rng.normal(0, 1, 500), index=dates, name='VIX')

fb = FeatureBuilder()
asset_feats = fb.asset_features(prices, market)
global_feats = fb.global_features(prices, market, vix)
print('asset_features shape:', asset_feats.shape, '(N, T, F_asset)')
print('global_features shape:', global_feats.shape, '(T, F_global)')

## 2. Reward decomposition (Eq. 16 / 17)

$$r_t = 1000\,\tilde r_{p,t} - \lambda_{TO}\,TO_t\cdot 100 - \lambda_{conc}(\mathrm{HHI}_t - \mathrm{HHI}_t^{\min})\cdot 100$$

with $\tilde r_{p,t}=\log(1+r_{p,t}^{net})$, $TO_t=\sum_i|w_{i,t}-w_{i,t^-}|$, $\mathrm{HHI}_t=\sum_i w_i^2$, $\mathrm{HHI}_t^{\min}=1/N_t$.

Defaults from the paper: $\lambda_{TO}=0.003$, $\lambda_{conc}\in\{0,0.1,0.5\}$.

In [ ]:
def reward_decompose(r_net, turnover, hhi, n_assets, lam_to=0.003, lam_c=0.1):
    log_term = 1000 * np.log1p(r_net)
    to_penalty = lam_to * turnover * 100
    conc_penalty = lam_c * (hhi - 1.0 / n_assets) * 100
    return log_term, -to_penalty, -conc_penalty

lt, pt, pc = reward_decompose(r_net=0.003, turnover=0.5, hhi=0.15, n_assets=21)
print(f'log_return term : {lt:+.4f}')
print(f'turnover penalty: {pt:+.4f}')
print(f'conc.  penalty  : {pc:+.4f}')
print(f'reward total    : {lt+pt+pc:+.4f}')

## 3. SAC training on synthetic data (smoke test)

Mirrors the paper's SAC hyperparameters (Table 4) but on **synthetic** data with a tiny budget (200 env steps) so it runs in seconds on CPU.

In [ ]:
cfg = load_config(REPO / 'configs' / 'lstm_2_ndx.yaml')
cfg['device'] = 'cpu'  # force CPU for the demo
cfg['data']['topk'] = 10
cfg['data']['lookback_window'] = 30
cfg['model']['lstm_hidden'] = 32
print('Config loaded:', cfg['model']['encoder_type'], '+', cfg['model']['policy_type'])

In [ ]:
from portfolio_rl.envs import PortfolioEnv
from portfolio_rl.agents import SACAgent

n_days, k = 400, cfg['data']['topk']
lookback = cfg['data']['lookback_window']
af = rng.standard_normal((n_days, k, 15)).astype(np.float32) * 0.5
gf = rng.standard_normal((n_days, 7)).astype(np.float32) * 0.3
ar = rng.normal(0.0005, 0.012, size=(n_days, k)).astype(np.float32)
br = ar.mean(axis=1)

env = PortfolioEnv(
    asset_features=af, global_features=gf,
    asset_returns=ar, benchmark_returns=br,
    lookback_window=lookback,
    transaction_cost_bps=cfg['data']['transaction_cost_bps'],
    lambda_turnover=cfg['evaluation']['lambda_turnover'],
    lambda_concentration=cfg['evaluation']['lambda_concentration'],
    reward_type=cfg['evaluation']['reward_type'],
    allow_cash=cfg['model']['allow_cash'],
)
agent = SACAgent(cfg, env, torch.device('cpu'))
print('SAC agent built; encoder out_dim =', agent.encoder.out_dim)

In [ ]:
obs, _ = env.reset()
rewards, weights_log, returns_log = [], [], []
for step in range(200):
    action = agent.select_action(obs)
    next_obs, r, term, trunc, info = env.step(action)
    agent.buffer.push(obs, action, r, next_obs, term or trunc)
    agent.update()
    rewards.append(r)
    weights_log.append(info['weights'])
    returns_log.append(info['portfolio_return'])
    obs = env.reset()[0] if (term or trunc) else next_obs

print(f'mean reward       : {np.mean(rewards):+.4f}')
print(f'mean daily return : {np.mean(returns_log):+.5f}')
print(f'final cash weight : {weights_log[-1][-1]:.3f}')

## 4. Performance metrics (paper §5.8)

In [ ]:
from portfolio_rl.evaluation import PerformanceMetrics

ret_series = pd.Series(returns_log)
metrics = PerformanceMetrics.compute_all(ret_series)
for k_, v in metrics.items():
    print(f'{k_:7s}: {v:.4f}')

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(8, 5), sharex=True)
ax[0].plot(np.cumsum(rewards))
ax[0].set_ylabel('cumulative reward')
ax[0].set_title('SAC training (synthetic data)')
equity = (1 + ret_series).cumprod()
ax[1].plot(equity.values)
ax[1].set_ylabel('equity curve (synthetic OOS)')
ax[1].set_xlabel('trading day')
plt.tight_layout(); plt.show()

## 5. Next steps

- Drop historical index-membership CSVs into `data/membership/{ndx,nky,sx5e}/` (see README).
- Run `bash data/download.sh` to pull yfinance prices for 2003-01-02 → 2026-03-13.
- Launch the full WFO grid:
  ```bash
  python scripts/run_walk_forward.py --config configs/lstm_2_ndx.yaml --num-folds 16
  ```
- Repeat for the remaining four configurations × three markets = 15 runs.

---
**Reproduction confidence:** SIR 0.88 · Plan 0.85 · Code 0.78 · Notebook 0.80.
Main residual unknowns: equity/cash split distribution for `LSTM_2`, exact attention variant, critic MLP depth, and (most critically) **historical index membership** which is paywalled.